## Q2

In [ ]:
import pymc as pm
import arviz as az
import pandas as pd
import numpy as np
import pytensor.tensor as pt

In [11]:
lamm = pd.read_csv("LAMM.csv")
lamm.head()

,x1,x2,x3,x4,y
0,10,10,0,100,12.08
1,10,10,5,100,12.73
2,10,10,10,100,14.13
3,10,50,0,100,13.12
4,10,50,5,100,11.84


In [12]:
X = lamm.iloc[:, 0:-1].to_numpy()
y = lamm.iloc[:, -1].to_numpy()

In [ ]:
with pm.Model() as model:
    X_data = pm.Data("X", X)

    beta0 = pm.Normal("beta0", 1.36, 1)
    beta1 = pm.Normal("beta1", 0.89, 1)
    beta2 = pm.Normal("beta2", 0.0014, 1)
    beta3 = pm.Normal("beta3", 0.0268, 1)
    beta4 = pm.Normal("beta4", 0.0034, 1)

    var = pm.InverseGamma("var", 48, 23)

    mu = (beta0 * (pt.math.pow(X_data[:, 0], beta1))) * pt.math.exp(
        (X_data[:, 1] * beta2)
        - (X_data[:, 2] * beta3) * pt.math.exp(X_data[:, 3] * -beta4)
    )
    pm.Normal("lik", mu, pt.math.sqrt(var), observed=y)

    trace = pm.sample(3000, init="adapt_diag")

print(az.summary(trace, hdi_prob=0.95))

pm.set_data({"X": np.array([10, 50, 5, 200]).reshape(1, -1)}, model=model)
ppc = pm.sample_posterior_predictive(trace, model=model, predictions=True)
print(az.summary(ppc.predictions, hdi_prob=0.95).mean())

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta0, beta1, beta2, beta3, beta4, var]


Output()

/Users/aaron/miniforge3/envs/pymc_f25_2/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:316: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)
/Users/aaron/miniforge3/envs/pymc_f25_2/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:316: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


/Users/aaron/miniforge3/envs/pymc_f25_2/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:316: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 1_000 tune and 3_000 draw iterations (4_000 + 12_000 draws total) took 10 seconds.
There were 5671 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Sampling: [lik]


Output()

        mean     sd  hdi_2.5%  hdi_97.5%  mcse_mean  mcse_sd  ess_bulk  \
beta0  1.567  0.102     1.361      1.776      0.002    0.008    2764.0   
beta1  0.887  0.022     0.840      0.928      0.000    0.002    3211.0   
beta2  0.001  0.000    -0.000      0.001      0.000    0.000      87.0   
beta3  0.132  0.931    -1.836      1.852      0.065    0.043     207.0   
beta4  0.702  0.606     0.039      1.866      0.119    0.010      11.0   
var    0.890  0.108     0.713      1.114      0.018    0.001      37.0   

       ess_tail  r_hat  
beta0    2621.0   1.13  
beta1    2551.0   1.14  
beta2    3925.0   1.03  
beta3    3627.0   1.02  
beta4      13.0   1.28  
var       908.0   1.07  


mean            12.373521
sd               0.966729
hdi_2.5%        10.482708
hdi_97.5%       14.269896
mcse_mean        0.009604
mcse_sd          0.006354
ess_bulk     10088.541667
ess_tail     11142.145833
r_hat            1.000000
dtype: float64


In [15]:
def transform_mean(mu):
    """
    Transform Normal mean to LogNormal mean to preserve that information in our priors
    """
    tau = 1
    return np.log(mu) - (1 / (2 * tau))

In [16]:
with pm.Model() as model:
    X_data = pm.Data("X", X, mutable=True)

    beta0 = pm.LogNormal("beta0", transform_mean(1.36), 1)
    beta1 = pm.LogNormal("beta1", transform_mean(0.89), 1)
    beta2 = pm.LogNormal("beta2", transform_mean(0.0014), 1)
    beta3 = pm.LogNormal("beta3", transform_mean(0.0268), 1)
    beta4 = pm.LogNormal("beta4", transform_mean(0.0034), 1)

    var = pm.InverseGamma("var", 48, 23)

    mu = (beta0 * (pt.math.pow(X_data[:, 0], beta1))) * pt.math.exp(
        (X_data[:, 1] * beta2)
        - (X_data[:, 2] * beta3) * pt.math.exp(X_data[:, 3] * -beta4)
    )
    pm.Normal("lik", mu, pt.math.sqrt(var), observed=y)

    trace = pm.sample(3000)

az.summary(trace, hdi_prob=0.95)

/Users/aaron/miniforge3/envs/pymc_f25_2/lib/python3.13/site-packages/pymc/data.py:384: FutureWarning: Data is now always mutable. Specifying the `mutable` kwarg will raise an error in a future release
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta0, beta1, beta2, beta3, beta4, var]


Output()

Sampling 4 chains for 1_000 tune and 3_000 draw iterations (4_000 + 12_000 draws total) took 8 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta0,1.609,0.109,1.396,1.822,0.002,0.001,5161.0,6010.0,1.0
beta1,0.888,0.022,0.846,0.934,0.000,0.000,5185.0,6137.0,1.0
beta2,0.001,0.000,0.000,0.001,0.000,0.000,6955.0,4840.0,1.0
beta3,0.010,0.003,0.005,0.016,0.000,0.000,6737.0,6978.0,1.0
beta4,0.002,0.002,0.000,0.005,0.000,0.000,6545.0,6629.0,1.0
var,0.781,0.093,0.609,0.966,0.001,0.001,9397.0,7355.0,1.0
